# Notebook 03: Gold Dimensions
## Propósito
Construir las 5 dimensiones del modelo estrella:
- dim date (conformada, calendario continuo)
- dim_seller (SCD tipo 1)
- dim_product (SCD tipo 1, con campos derivados)
- dim_customer (SCD tipo 2, atributos city/state trackeados)
- dim_order_junk (junk dimension)

## Inputs
tablas "silver_*" en {SILVER_LAKEHOUSE}
## Outputs
Tablas "dim_*" en {GOLD_LAKEHOUSE}

In [3]:
# =================================================================
# PARÁMETROS DEL NOTEBOOK
# =================================================================
# Estos valores se pueden sobrescribir cuando el notebook es invocado
# desde un Data Pipeline. Si se corre manualmente, usa estos defaults.
# -----------------------------------------------------------------

SILVER_LAKEHOUSE = "lh_olist_silver"
GOLD_LAKEHOUSE = "lh_olist_gold"
WRITE_MODE = "overwrite"

print(f"Silver (source): {SILVER_LAKEHOUSE}")
print(f"Gold (target): {GOLD_LAKEHOUSE}")

StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 5, Finished, Available, Finished, False)

Silver (source): lh_olist_silver
Gold (target): lh_olist_gold


In [4]:
# **Celda 3 — Imports:**

from pyspark.sql.functions import (col, expr, when, lit, coalesce, 
    year, month, dayofmonth, dayofweek, weekofyear, quarter, 
    date_format, current_timestamp, 
    sha2, concat_ws, monotonically_increasing_id, 
    round as spark_round, max as spark_max, min as spark_min)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, BooleanType, TimestampType, DataType

StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 6, Finished, Available, Finished, False)

In [5]:
# **Celda 4 — Helpers:**

def read_silver(table_name: str):
    return spark.read.table(f"{SILVER_LAKEHOUSE}.{table_name}")

def write_gold(df, table_name: str):
    (df.write.format("delta")
        .mode(WRITE_MODE)
        .option("overwriteSchema", "true")
        .saveAsTable(f"{GOLD_LAKEHOUSE}.{table_name}"))
    print(f"{GOLD_LAKEHOUSE}.{table_name}: {df.count():,} filas")

def read_gold(table_name: str):
    "solo para validar dentro del mismo notebook"
    return spark.read.table(f"{GOLD_LAKEHOUSE}.{table_name}")

StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 7, Finished, Available, Finished, False)

In [6]:
# **Celda 5 — dim_date (conformada, continua):**

print("dim_date (conformada)")

orders = read_silver("silver_orders")
reviews = read_silver("silver_order_reviews")

#a partir de las columnas de cada tabla sacamos minimo, maximo etc de fecha
#.collect() → trae el resultado al driver como una lista de filas.
#[0][0] → extrae el unico valor de la unica fila **Row(fecha_min='2017-12-05')**.

fecha_min = orders.agg(spark_min("order_purchase_timestamp").cast("date")).collect()[0][0]
fecha_max_o = orders.agg(spark_max("order_purchase_timestamp").cast("date")).collect()[0][0]
fecha_max_r = reviews.agg(spark_max("review_creation_date").cast("date")).collect()[0][0]
fecha_max = max(fecha_max_o, fecha_max_r) if fecha_max_r else fecha_max_o

print(f"Rango: {fecha_min} --- {fecha_max}")

df_rango = spark.sql(f"""
    --Genera todas las fechas entre fecha_min y fecha_max (de 1 día en 1 día) y las expande para obtener una fila por cada fecha del rango.
    SELECT explode(sequence(DATE '{fecha_min}', DATE '{fecha_max}', INTERVAL 1 DAY)) AS fecha
""")

dim_date = (df_rango
    .withColumn("date_sk", date_format(col("fecha"), "yyyyMMdd").cast(IntegerType()))
    .withColumn("anio", year(col("fecha")))
    .withColumn("trimestre", quarter(col("fecha")))
    .withColumn("mes_numero", month(col("fecha")))
    .withColumn("mes_nombre", date_format(col("fecha"), "MMMM"))
    .withColumn("mes_anio", date_format(col("fecha"), "yyyy-MM"))
    .withColumn("semana_anio", weekofyear(col("fecha")))
    .withColumn("dia_mes", dayofmonth(col("fecha")))
    .withColumn("dia_semana_num", dayofweek(col("fecha")))
    .withColumn("dia_nombre", date_format(col("fecha"), "EEEE"))
    .withColumn("es_fin_semana",
        when(col("dia_semana_num").isin([1,7]), True).otherwise(False))
    #consolidar tabla
    .select("date_sk", "fecha", "anio", "trimestre", "mes_numero", "mes_nombre",
            "mes_anio", "semana_anio", "dia_mes", "dia_semana_num", "dia_nombre",
            "es_fin_semana"))

write_gold(dim_date, "dim_date")

StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 8, Finished, Available, Finished, False)

dim_date (conformada)
Rango: 2016-09-04 --- 2018-10-17
lh_olist_gold.dim_date: 774 filas


In [7]:
# **Celda 6 — dim_seller (SCD1):**

print("dim_seller (SCD Tipo 1)")

dim_seller = (read_silver("silver_sellers")
    .withColumn("seller_sk", monotonically_increasing_id())
    .select(
        "seller_sk",
        col("seller_id").alias("seller_natural_id"),
        col("seller_zip_code_prefix").alias("zip_prefix"),
        col("seller_city").alias("city"),
        col("seller_state").alias("state")
    ))

write_gold(dim_seller,"dim_seller")

StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 9, Finished, Available, Finished, False)

dim_seller (SCD Tipo 1)
lh_olist_gold.dim_seller: 3,095 filas


In [8]:
# **Celda 7 — dim_product (SCD1):**

print("dim_product (SCD Tipo 1)")

silver_products = read_silver("silver_products")
#Calculamos la mediana del peso de los productos (product_weight_g) para obtener un valor representativo del peso típico y poder utilizarlo
median_weight = silver_products.approxQuantile("product_weight_g", [0.5],0.01)[0]

dim_product = (silver_products
    .withColumn("product_sk", monotonically_increasing_id())
    #Reemplaza los valores null de product_weight_g por la mediana previamente calculada (median_weight) mediante coalesce() y lit().
    .withColumn("product_weight_g", coalesce(col("product_weight_g"), lit(median_weight)))
    .withColumn("product_volume_cm3",
        spark_round(col("product_length_cm") * col("product_height_cm") * col("product_width_cm"), 2))
    .withColumn("size_category",
        when(col("product_volume_cm3") < 1000, "Small")
        .when(col("product_volume_cm3") < 10000, "Medium")
        .when(col("product_volume_cm3") < 100000, "Large")
        .otherwise("Extra Large"))
    .select(
        "product_sk",
        col("product_id").alias("producto_natural_id"),
        col("product_category_name_english").alias("category"),
        "product_weight_g",
        "product_volume_cm3",
        "size_category"
    ))
    
write_gold(dim_product, "dim_product")


StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 10, Finished, Available, Finished, False)

dim_product (SCD Tipo 1)
lh_olist_gold.dim_product: 32,951 filas


In [9]:
# **Celda 8 — dim_customer (SCD Tipo 2):**

print("dim_customer (SCD Tipo 2)")

silver_customers = read_silver("silver_customers")
silver_orders = read_silver("silver_orders")

#1. (customer + timestamp de evento, orden asociada)
customer_with_date = (silver_customers
    .join(silver_orders.select("customer_id", "order_purchase_timestamp"),
            on = "customer_id", how = "inner")
    .select(
        "customer_unique_id",
        "customer_city",
        "customer_state",
        "customer_zip_code_prefix",
        col("order_purchase_timestamp").alias("event_timestamp")
    ))

#2. hash de los atributos trackeados
customer_hashed = customer_with_date.withColumn(
    "attribute_hash",
    sha2(concat_ws("|",
    coalesce(col("customer_city"), lit("")),
    coalesce(col("customer_state"), lit("")),
    coalesce(col("customer_zip_code_prefix").cast("string"), lit(""))
    ), 256)
)

#3. Con un función de ventana, detecta versiones nuevas (cambio de hash respecto a la versión anterior)
customer_with_lag = customer_hashed.withColumn(
    "previous_hash",
    expr("LAG(attribute_hash) OVER (PARTITION BY customer_unique_id ORDER BY event_timestamp)")
)

version_starts = customer_with_lag.filter(
#El último estado en el tiempo para las versiones (el último registro o la última versión)
    col("previous_hash").isNull() | (col("attribute_hash") != col("previous_hash"))
)

print(f"  Customer events totales: {customer_hashed.count():,}")
print(f"  Versiones únicas detectadas: {version_starts.count():,}")

#4. calcular el valid_from / valid_to

dim_customer_scd2 = (version_starts
    .withColumn("valid_from", col("event_timestamp"))
    .withColumn("valid_to",
        coalesce(
            expr("LEAD(event_timestamp) OVER (PARTITION BY customer_unique_id ORDER BY event_timestamp)"),
            lit("9999-12-31 23:59:59").cast(TimestampType())
        ))
    .withColumn("is_current",
        when(col("valid_to") == lit("9999-12-31 23:59:59").cast(TimestampType()), True)
        .otherwise(False))
    .withColumn("customer_sk", monotonically_increasing_id())
    .select(
        "customer_sk",
        col("customer_unique_id").alias("customer_natural_id"),
        col("customer_city").alias("city"),
        col("customer_state").alias("state"),
        col("customer_zip_code_prefix").alias("zip_prefix"),
        "attribute_hash",
        "valid_from",
        "valid_to",
        "is_current"
    ))

write_gold(dim_customer_scd2, "dim_customer")


StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 11, Finished, Available, Finished, False)

dim_customer (SCD Tipo 2)
  Customer events totales: 99,441
  Versiones únicas detectadas: 96,355
lh_olist_gold.dim_customer: 96,355 filas


In [11]:
#**Celda 9 — Validación SCD2:**

dc = read_gold("dim_customer")

#BK (business key): se repite - identifica al cliente real
#SK (surrogate key): único - identifica cada versión del registro
#múltiples SK pueden existir para un mismo BK si hay historial

clientes_con_historia = (dc.groupBy("customer_natural_id")
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc()))

total_con_historia = clientes_con_historia.count()
print(f"Clientes con múltiples versiones: {total_con_historia:,}")

if total_con_historia > 0:
    ejemplo_id = clientes_con_historia.first()["customer_natural_id"]
    print(f"Historia del cliente {ejemplo_id}")
    (dc.filter(col("customer_natural_id") == ejemplo_id)
        .orderBy("valid_from")
        .select("customer_sk", "city", "state", "zip_prefix", "valid_from", "valid_to", "is_current")
        .show(truncate = False))
else:
    print("Ningun cliente tiene historial, validar atributos de hash")

StatementMeta(, 341793ab-098f-450f-b9b9-586840108eac, 13, Finished, Available, Finished, False)

Clientes con múltiples versiones: 252
Historia del cliente 3e43e6105506432c953e165fb2acf44c
+-----------+------------+-----+----------+-------------------+-------------------+----------+
|customer_sk|city        |state|zip_prefix|valid_from         |valid_to           |is_current|
+-----------+------------+-----+----------+-------------------+-------------------+----------+
|17179871045|praia grande|SP   |11701     |2017-09-18 18:53:15|2017-12-01 09:30:36|false     |
|17179871046|praia grande|SP   |11700     |2017-12-01 09:30:36|2018-01-11 10:56:15|false     |
|17179871047|praia grande|SP   |11704     |2018-01-11 10:56:15|2018-02-12 10:12:54|false     |
|17179871048|praia grande|SP   |11701     |2018-02-12 10:12:54|9999-12-31 23:59:59|true      |
+-----------+------------+-----+----------+-------------------+-------------------+----------+



In [10]:
#**Celda 10 — dim_order_junk:**

print("dim_order_junk")

silver_orders = read_silver("silver_orders")
silver_payments = read_silver("silver_order_payments")
silver_reviews = read_silver("silver_order_reviews")

# pago principal por orden
payment_main = (silver_payments
    .filter(col("payment_sequential") == 1)
    .select("order_id", "payment_type", "installments_bucket"))

# flag la orden tiene review
order_with_review = (silver_reviews
    .select("order_id").distinct()
    .withColumn("has_review", lit(True)))

#combinar atributos por orden
order_attrs = (silver_orders
    .join(payment_main, on = "order_id", how = "left")
    .join(order_with_review, on = "order_id", how = "left")
    .withColumn("has_review", coalesce(col("has_review"), lit(False)))
    .withColumn("payment_type", coalesce(col("payment_type"), lit("none")))
    .withColumn("installments_bucket", coalesce(col("installments_bucket"), lit("none")))
    .select("order_status", "delivery_status_bucket", "payment_type", "installments_bucket", "has_review"))

dim_order_junk = (order_attrs.distinct()
    .withColumn("junk_sk", monotonically_increasing_id())
    .withColumn("attribute_hash",
        sha2(concat_ws("|",
            col("order_status"),
            col("delivery_status_bucket"),
            col("payment_type"),
            col("installments_bucket"),
            col("has_review").cast("string")
        ), 256))
    .select("junk_sk", "order_status", "delivery_status_bucket",
            "payment_type", "installments_bucket", "has_review", "attribute_hash"
    ))

write_gold(dim_order_junk, "dim_order_junk")
print(f"Combinaciones unicas: {read_gold('dim_order_junk').count():,}")
read_gold("dim_order_junk").show(10, truncate=False)

StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 12, Finished, Available, Finished, False)

dim_order_junk
lh_olist_gold.dim_order_junk: 107 filas
Combinaciones unicas: 107
+-------+------------+----------------------+------------+-------------------+----------+----------------------------------------------------------------+
|junk_sk|order_status|delivery_status_bucket|payment_type|installments_bucket|has_review|attribute_hash                                                  |
+-------+------------+----------------------+------------+-------------------+----------+----------------------------------------------------------------+
|0      |shipped     |Not Delivered         |voucher     |1                  |true      |4ce0250bd895ad3648c87e9670240206a97ea184472b6f495a7436e0edcb5d36|
|1      |delivered   |On time               |credit_card |6-12               |false     |56d24163a590174467734a7880a791562999516d97cc0ab2fc7a8f1ac47350a7|
|2      |delivered   |On time               |credit_card |1                  |false     |f0d885b6d12e2566f2d512d2a4f759f03ca58c3ee82e15c9394a119

## Que es una junk dimension?

Una **junk dimension** agrupa varios atributos categóricos pequeños (baja cardinalidad) en una sola tabla de dimensión, donde cada fila representa una **combinación única** de esos atributos que aparece en los datos reales.

**Por qué hacerla:** evita ensuciar el fact con muchas columnas planas o llenar el modelo con mini-dimensiones de 2-5 filas cada una. Una sola FK en el fact (`junk_sk`) reemplaza N columnas o N joins.

**En este proyecto:** combinamos 5 atributos de la orden — `order_status`, `delivery_status_bucket`, `payment_type`, `installments_bucket`, `has_review` — en `dim_order_junk`. El producto cartesiano teórico es ~1024 combinaciones, pero solo persistimos las que existen en los datos (107). Las demás son combinaciones imposibles o que nunca ocurrieron.

**Regla:** si un atributo es pequeño, descriptivo, sin jerarquía propia, y no merece una dimensión solo, va a la junk.

In [13]:
# **Celda 11 — Validación final del notebook:**

print("=" * 60)
print("VALIDACIÓN GOLD DIMENSIONES")
print("=" * 60)

DIMS = ["dim_date", "dim_seller", "dim_product", "dim_customer", "dim_order_junk"]

for d in DIMS:
    count = read_gold(d).count()
    print(f"{d}: {count:,}")

StatementMeta(, fc20613d-39b8-4d18-817e-8263dd81d5ab, 15, Finished, Available, Finished, False)

VALIDACIÓN GOLD DIMENSIONES
dim_date: 774
dim_seller: 3,095
dim_product: 32,951
dim_customer: 96,355
dim_order_junk: 107
